In [1]:
import sys; print(sys.executable)

/home/hasnain/projects/velo-disponibility/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


In [3]:
df = pd.read_csv('../data/raw/readings.csv')
df.head()

,fetched_at,lastupdate,idstation,nom,lon,lat,etat,slots_total,slots_free,bikes,consistent
0,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5505,Sainte-Anne,-1.680534,48.114219,En fonctionnement,24,13,11,1
1,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5509,Saint-Georges Piscine,-1.674417,48.112385,En fonctionnement,18,16,2,1
2,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5510,Musée Beaux-Arts,-1.674080,48.109601,En fonctionnement,15,1,14,1
3,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5514,Bonnets Rouges,-1.665814,48.106846,En fonctionnement,23,18,5,1
4,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5517,Charles de Gaulle,-1.677119,48.105111,En fonctionnement,24,8,16,1


# 1. Filter active stations and valid records

In [5]:
df = df[df['etat'] == 'En fonctionnement'].copy()
df = df[df['consistent'] == 1]

# 2. Check Duplicates

In [6]:
print(df.duplicated(["idstation", "lastupdate"]).sum())

0


# 3. Parse timestamps

In [7]:
df['lastupdate'] = pd.to_datetime(df['lastupdate'])

# 4. Create target classes

Bike-count classes: 0, 1-2, 3-5, 6-15, 16-19, 20-25, and 26+ bikes.

In [8]:
# Bike-count classes: 0, 1-2, 3-5, 6-15, 16-19, 20-25, and 26+
bins = [-1, 0, 2, 5, 15, 19, 25, float('inf')]
labels = [0, 1, 2, 3, 4, 5, 6]

# Apply directly to columns
df['bike_bin'] = pd.cut(df['bikes'], bins=bins, labels=labels).astype(int)

# 5. Feature engineering: Extract temporal features

In [9]:
df['ts'] = pd.to_datetime(df['lastupdate'], utc=True).dt.tz_convert('Europe/Paris')
df['hour'] = df['ts'].dt.hour
df['day_of_week'] = df['ts'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

/tmp/ipykernel_140754/3469264913.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['ts'].dt.dayofweek


# 6. Select features for ML

In [10]:
feature_cols = ['idstation', 'lat', 'lon', 'slots_total', 'hour', 'day_of_week', 'is_weekend']
target_col = ['bike_bin']

In [11]:
print(df['bike_bin'].value_counts())
print(df['bike_bin'].value_counts(normalize=True).round(2))

bike_bin
3    6871
2    2805
1    1484
4     842
0     535
5     347
6      48
Name: count, dtype: int64
bike_bin
3    0.53
2    0.22
1    0.11
4    0.07
0    0.04
5    0.03
6    0.00
Name: proportion, dtype: float64


In [12]:
train, test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['bike_bin']
)

In [13]:
def mode(s): return s.mode().iloc[0]

# A: always the most common class
major = train['bike_bin'].mode()[0]
pred_a = pd.Series(major, index=test.index)

# B: most common class per station + hour + weekday, with fallbacks
keys = [['idstation','hour','day_of_week'], ['idstation','hour'], ['idstation']]
pred_b = pd.Series(np.nan, index=test.index)
for k in keys:
    table = train.groupby(k)['bike_bin'].agg(mode).rename('p').reset_index()
    m = test[k].merge(table, on=k, how='left')
    m.index = test.index
    pred_b = pred_b.fillna(m['p'])
pred_b = pred_b.fillna(major).astype(int)

for name, p in [('majority', pred_a), ('station-hour-weekday', pred_b)]:
    print(name, accuracy_score(test['bike_bin'], p),
          f1_score(test['bike_bin'], p, average='macro'))

majority 0.5315036722071897 0.09915627028196437
station-hour-weekday 0.6397371472748358 0.45896535599847194


In [14]:
df
train

,fetched_at,lastupdate,idstation,nom,lon,lat,etat,slots_total,slots_free,bikes,consistent,bike_bin,ts,hour,day_of_week,is_weekend
2921,2026-10-12T13:20:00+00:00,2026-10-12 13:18:08+00:00,5580,Painlevé,-1.659762,48.123463,En fonctionnement,29,14,15,1,3,2026-10-12 15:18:08+02:00,15,0,0
10522,2026-10-19T18:40:00+00:00,2026-10-19 18:38:08+00:00,5590,Bourgeois,-1.655278,48.098086,En fonctionnement,24,8,16,1,4,2026-10-19 20:38:08+02:00,20,0,0
3516,2026-10-13T10:40:00+00:00,2026-10-13 10:38:09+00:00,5512,TNB,-1.673711,48.107748,En fonctionnement,28,27,1,1,1,2026-10-13 12:38:09+02:00,12,1,0
7415,2026-10-15T13:40:00+00:00,2026-10-15 13:38:07+00:00,5519,Colombier,-1.680454,48.106045,En fonctionnement,24,20,4,1,2,2026-10-15 15:38:07+02:00,15,3,0
12274,2026-10-22T11:22:37+00:00,2026-10-22 11:21:08+00:00,5591,Mabilais,-1.693098,48.105108,En fonctionnement,25,9,16,1,4,2026-10-22 13:21:08+02:00,13,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7673,2026-10-15T15:20:00+00:00,2026-10-15 15:18:08+00:00,5592,Gares,-1.673233,48.104097,En fonctionnement,50,31,19,1,4,2026-10-15 17:18:08+02:00,17,3,0
4174,2026-10-13T15:00:00+00:00,2026-10-13 14:58:08+00:00,5569,Champs Manceaux,-1.682284,48.091114,En fonctionnement,17,3,14,1,3,2026-10-13 16:58:08+02:00,16,1,0
8490,2026-10-16T17:20:00+00:00,2026-10-16 17:17:08+00:00,5579,Les Gayeulles,-1.657528,48.129454,En fonctionnement,30,25,5,1,2,2026-10-16 19:17:08+02:00,19,4,0
1074,2026-10-10T15:00:00+00:00,2026-10-10 14:58:08+00:00,5547,Beaulieu Chimie,-1.633875,48.116458,En fonctionnement,30,19,11,1,3,2026-10-10 16:58:08+02:00,16,5,1


In [15]:
from pathlib import Path

output_dir = Path('../data')
output_dir.mkdir(parents=True, exist_ok=True)
df.to_parquet(output_dir / 'clean.parquet', index=False)
train.to_parquet(output_dir / 'train.parquet', index=False)
test.to_parquet(output_dir / 'test.parquet', index=False)
print(f'Saved {len(df):,} cleaned rows, {len(train):,} train rows, and {len(test):,} test rows.')

Saved 12,932 cleaned rows, 10,345 train rows, and 2,587 test rows.
